In [ ]:
# Setup: clone the StarX repo (house pattern), mount Drive.
import os
import subprocess
import sys

BRANCH = "main"
TRIPOSR_COMMIT = "107cefdc244c39106fa830359024f6a2f1c78871"
NOTEBOOK_ID = "08"

IN_COLAB = os.path.exists("/content")
if IN_COLAB:
    REPO_DIR = "/content/StarX"
    if not os.path.exists(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--branch", BRANCH,
             "https://github.com/SattamAltwaim/StarX.git", REPO_DIR],
            check=True,
        )
else:
    REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from starx import pins

assert pins.TRIPOSR_COMMIT == TRIPOSR_COMMIT, "notebook pin out of sync with starx/pins.py"
if pins.PIP_PINS[NOTEBOOK_ID]:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *pins.PIP_PINS[NOTEBOOK_ID]],
        check=True,
    )

from starx import colab as scolab

DRIVE = scolab.mount_drive()
print("Drive mounted at:", DRIVE)

# 08 - Move the data from Drive to Ibex

This notebook pushes the processed StarX data from your Google Drive to a KAUST Ibex filesystem, so training runs there instead of Colab. It runs **in Colab** (where Drive is already mounted) and sends files over rsync+ssh:

1. pick what to send (shards, smoke shards, run checkpoints, stats),
2. generate a throwaway SSH key in this VM and authorize it on Ibex with one paste - no password ever enters this notebook,
3. rsync everything across (resumable - re-run the cell after any disconnect),
4. verify file counts match, then follow the last section to launch notebook 05 on an Ibex GPU node.

Any Colab runtime works; no GPU needed here.

In [ ]:
# Configuration - every tunable for this notebook lives here.
import subprocess
from pathlib import Path

IBEX_USER = ""                              # your KAUST/Ibex username - REQUIRED
IBEX_HOST = "ilogin.ibex.kaust.edu.sa"      # login or data-transfer node
IBEX_DEST = f"/ibex/user/{IBEX_USER}/StarX" # where the data lands on Ibex

INCLUDE_SHARDS = True      # shards/train + shards/test (the training data)
INCLUDE_SMOKE = True       # the 20-design smoke shards (small)
INCLUDE_RUNS = True        # checkpoints, logs, val grids of existing runs
INCLUDE_STATS = True       # splits.json and dataset stats (tiny)
INCLUDE_RAW_ZIP = False    # the 2 GB dataset zip - only needed for
                           # notebooks 01-03 and 06's GT meshes on Ibex

DRIVE_ROOT = Path("/content/drive/MyDrive/StarX")

assert IN_COLAB, "this notebook pushes FROM Drive - run it in Colab"
assert IBEX_USER, "set IBEX_USER in this cell first"
assert DRIVE_ROOT.exists(), "Drive is mounted but has no StarX folder"
print(f"{IBEX_USER}@{IBEX_HOST}  ->  {IBEX_DEST}")

In [ ]:
# Inventory: what will be sent, per the toggles above.
SELECTION = {
    "shards/train": INCLUDE_SHARDS,
    "shards/test": INCLUDE_SHARDS,
    "shards/smoke_train": INCLUDE_SMOKE,
    "shards/smoke_test": INCLUDE_SMOKE,
    "shards/smoke": INCLUDE_SMOKE,
    "stats": INCLUDE_STATS,
    "runs": INCLUDE_RUNS,
    "raw": INCLUDE_RAW_ZIP,
}

transfer_dirs = []
total_bytes = 0
print(f"{'directory':24s} {'files':>7s} {'size':>10s}  status")
for rel, include in SELECTION.items():
    path = DRIVE_ROOT / rel
    if not include:
        print(f"{rel:24s} {'-':>7s} {'-':>10s}  skipped (toggle off)")
        continue
    if not path.exists():
        print(f"{rel:24s} {'-':>7s} {'-':>10s}  absent on Drive")
        continue
    files = [f for f in path.rglob("*") if f.is_file()]
    size = sum(f.stat().st_size for f in files)
    total_bytes += size
    transfer_dirs.append(rel)
    print(f"{rel:24s} {len(files):>7d} {size / 2**30:>9.2f}G  queued")
print(f"\ntotal to transfer: {total_bytes / 2**30:.2f} GiB "
      f"across {len(transfer_dirs)} directories")

In [ ]:
# One-time key setup: generate an ephemeral SSH keypair inside this Colab
# VM and print the single command to authorize it on Ibex. Your password
# is never typed here; the key dies with the VM.
KEY_PATH = Path("/content/ibex_ed25519")
if not KEY_PATH.exists():
    subprocess.run(
        ["ssh-keygen", "-t", "ed25519", "-N", "", "-q",
         "-f", str(KEY_PATH), "-C", "starx-colab-transfer"],
        check=True,
    )
public_key = Path(str(KEY_PATH) + ".pub").read_text().strip()
SSH_OPTS = [
    "-i", str(KEY_PATH),
    "-o", "StrictHostKeyChecking=accept-new",
    "-o", "BatchMode=yes",
]

print("Run this ONCE in a terminal on Ibex (ssh in normally first), then")
print("come back and run the next cell:\n")
print("mkdir -p ~/.ssh && chmod 700 ~/.ssh && \\")
print(f"echo '{public_key}' >> ~/.ssh/authorized_keys && \\")
print("chmod 600 ~/.ssh/authorized_keys")

In [ ]:
# Connectivity check: proves the key is authorized before any long
# transfer, and creates the destination directory. If this fails with
# "Permission denied", the public key from the previous cell has not been
# added on Ibex yet.
probe = subprocess.run(
    ["ssh", *SSH_OPTS, f"{IBEX_USER}@{IBEX_HOST}",
     f"mkdir -p {IBEX_DEST} && echo CONNECTED && hostname && df -h {IBEX_DEST} | tail -1"],
    capture_output=True, text=True, timeout=60,
)
print(probe.stdout)
if probe.returncode != 0:
    print(probe.stderr)
    raise RuntimeError(
        "ssh failed - add the public key on Ibex (previous cell), check "
        "IBEX_USER, and that you can reach the host from your network"
    )

In [ ]:
# The transfer. rsync is resumable: killing this cell and re-running it
# picks up where it stopped and skips anything already complete, so a
# Colab disconnect costs nothing. Reading from mounted Drive is the slow
# side - expect tens of MB/s.
transferred = []
for rel in transfer_dirs:
    print(f"\n=== {rel} ===")
    subprocess.run(
        ["ssh", *SSH_OPTS, f"{IBEX_USER}@{IBEX_HOST}", f"mkdir -p {IBEX_DEST}/{rel}"],
        check=True,
    )
    result = subprocess.run(
        ["rsync", "-a", "--partial", "--info=progress2",
         "-e", "ssh " + " ".join(SSH_OPTS),
         f"{DRIVE_ROOT}/{rel}/", f"{IBEX_USER}@{IBEX_HOST}:{IBEX_DEST}/{rel}/"],
    )
    if result.returncode == 0:
        transferred.append(rel)
        print(f"done: {rel}")
    else:
        print(f"rsync exited with {result.returncode} for {rel} - "
              "re-run this cell to resume")
print(f"\ncompleted {len(transferred)}/{len(transfer_dirs)} directories")

In [ ]:
# Verify: compare file counts and sizes between Drive and Ibex for every
# transferred directory.
print(f"{'directory':24s} {'local files':>12s} {'remote files':>13s} {'remote size':>12s}")
for rel in transferred:
    local_dir = DRIVE_ROOT / rel
    n_local = sum(1 for f in local_dir.rglob("*") if f.is_file())
    remote = subprocess.run(
        ["ssh", *SSH_OPTS, f"{IBEX_USER}@{IBEX_HOST}",
         f"find {IBEX_DEST}/{rel} -type f | wc -l; du -sh {IBEX_DEST}/{rel} | cut -f1"],
        capture_output=True, text=True,
    )
    if remote.returncode == 0:
        n_remote, size_remote = [s.strip() for s in remote.stdout.strip().splitlines()]
    else:
        n_remote, size_remote = "?", "?"
    marker = "" if str(n_local) == n_remote else "   MISMATCH - re-run the transfer cell"
    print(f"{rel:24s} {n_local:>12d} {n_remote:>13s} {size_remote:>12s}{marker}")
print("\nmatching counts everywhere = transfer complete")

## Running notebook 05 on Ibex

One-time setup in an Ibex terminal:

```
cd $HOME
git clone https://github.com/SattamAltwaim/StarX.git
mkdir -p $HOME/StarX/data
ln -s /ibex/user/$USER/StarX  $HOME/StarX/data/StarX
```

The notebooks resolve their storage root to `<repo>/data/StarX` when not on Colab, so the symlink points them at the transferred data.

Environment: activate a conda/venv that has a CUDA build of PyTorch - the setup cell pip-installs everything else pinned in `starx/pins.py` into that same environment. Then start Jupyter on a GPU node, for example:

```
srun --time=6:00:00 --gres=gpu:a100:1 --cpus-per-task=8 --mem=64G --pty bash
conda activate <your-torch-env>
cd $HOME/StarX/experiments
jupyter lab --no-browser --ip=$(hostname -i)
```

(check the KAUST Ibex documentation for the currently recommended Jupyter workflow and partition names; `--gres=gpu:v100:1` also works - the notebook picks fp16 there and bf16 on A100 automatically.)

In notebook 05, run the cells top to bottom through the resume cell, then the post-mortem cells (loss curves, gallery, before/after, probe) to analyze the existing `baseline_l4` run before launching any new training.

## Cleaning up

The private key lives only inside this Colab VM and disappears when the session ends. The public key you added on Ibex keeps working until you remove it - when you are done transferring, delete its line on Ibex:

```
grep -v starx-colab-transfer ~/.ssh/authorized_keys > ~/.ssh/ak.tmp && mv ~/.ssh/ak.tmp ~/.ssh/authorized_keys
```

Re-running this notebook later generates a fresh key and shows the paste command again.